In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
docs = loader.load()

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(docs)

In [4]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="bge-m3:latest",
)

In [5]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(texts, embeddings)

In [21]:
vectorstore.save_local("faiss_index")

In [6]:
query = "국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?"

In [7]:
results = vectorstore.similarity_search(query, k=3)

In [12]:
for result in results:
    print(result.page_content)
    print("-" * 30)

7.2 자격증 취득 지원 
직무 관련 국가 기술 자격 취득 시 축하금 및 수당을 지급합니다. 
자격 등급 
축하금 (1 회성)
자격 수당 (월)
대상 자격증 예시 
기술사/기능장
200 만원 
30 만원 
금속재료, 용접, 기계가공 등
기사 
50 만원 
10 만원 
일반기계, 전기, 산업안전 등
산업기사 
30 만원 
5 만원 
기계설계, 위험물 등 
기능사 
10 만원 
3 만원 
선반, 밀링, 특수용접 등 
 조건: 동일 등급 내 1 개 자격증만 수당 인정 (상위 등급 취득 시 갱신). 축하금은 횟수 
제한 없음. 
7.3 해외 연수 (Global Explorer) 
 대상: 연간 최우수 사원 (MVP) 및 우수 팀. 
 내용: 매년 10 월 독일/일본 등 선진 제조 현장 견학 및 문화 탐방 (7 박 9 일). 
 비용: 회사 전액 부담 (개인 경비 1,000 유로 별도 지급). 
 
8. 윤리 경영 및 보안 (Ethics & Security)
------------------------------
가족 수당 
부양 가족이 있는 자
배우자 5 만원, 
자녀당 3 만원 / 월 
등본 제출 필수 
4.4 특수 영입 및 장기 근속 보너스 (Signing Bonus) 
테크노빌드는 핵심 기술 인력 확보를 위해 입사 시점에 일시불로 지급되는 사이닝 보너스 
제도를 운영합니다. 
A. 지급 대상 및 금액 
1. 대상: 연구직 G3 이상 또는 사외 특급 기술 자격 보유자 중 채용위원회 승인을 득한 
자. 
2. 지급액: 전 직장 연봉의 10% ~ 50% 범위 내에서 개별 협의. 
3. 지급 시기: 입사 첫 달 급여일에 일괄 지급. 
B. 의무 근속 및 환수(Clawback) 조항 
지급 후 36 개월(3 년) 이내에 개인 사유로 퇴사할 경우, 미충족 기간에 대해 월할 계산하여 
반환해야 합니다. 
환수 금액= 기지급액×
(36 −실제근속개월수)
36
 
알림: 본인 귀책 사유에 의한 징계 해고 시에는 기간에 관계없이 전액 반환을 
원칙으로 합니다.
------

In [13]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트: {context}

질문: {question}
"""
)

In [14]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [15]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0,
)

In [16]:
chain = prompt | llm | parser

In [ ]:
response = chain.invoke({"context": results, "question": query})

In [18]:
response

'기사 자격증을 취득하면 50 만원 또는 10 만원을 받을 수 있습니다.'

In [19]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

gemini_llm = init_chat_model("google_genai:gemini-3.1-flash-lite")

gemini_chain = prompt | gemini_llm | parser

In [20]:
response = gemini_chain.invoke({"context": results, "question": query})
response

'제공된 문서에 따르면, 국가 기술 자격 중 기사 자격증을 취득할 경우 **축하금(1회성)으로 50만 원**을 받을 수 있으며, **월 10만 원의 자격 수당**을 받을 수 있습니다.'